# Descriptive summaries and monitoring coverage

Rebuild Results 3.2, Table S1, and the numerical summaries behind Figures 1 and 2.

**Step:** 04. Descriptive statistics (Results 3.2, Table S1).

**Article scope:** the main article and Supplementary Material.

**Inputs:** The current 23-home hourly and daily data, resolved through the sensor crosswalk.

**Outputs:** Seasonal summaries, equal-home diurnal profiles, amplitudes, threshold burdens, and Table S1.

**How to use this notebook:** select the Python environment described in `README.md`, restart the kernel,
and run the cells from top to bottom. Edit the configuration cell before running.
The checks below stop execution when the current data no longer match the manuscript specification.

## Locate the code folder and load the shared functions

The repository is found by searching this notebook's folder and its parent folders, so Jupyter can be started anywhere inside the repository. Set `INDOOR_AQ_CODE` only when it starts elsewhere.
It loads only the cells tagged `library` in `00_shared_functions.ipynb` (its demonstration cell is not executed).
Each notebook has its own imports and configuration, so it never depends on another notebook's kernel state.

In [ ]:
# Setup: find the code folder and load the shared functions.
from pathlib import Path
import json
import os
import sys
import types


def locate_code_directory():
    """Locate this repository (the folder with 00_shared_functions.ipynb) from the working directory or INDOOR_AQ_CODE."""
    configured = os.environ.get("INDOOR_AQ_CODE")
    if configured:
        candidates = [Path(configured).expanduser().resolve()]
    else:
        candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if ((candidate / "00_shared_functions.ipynb").is_file()
                and (candidate / "5_did_cits").is_dir()):
            return candidate.resolve()
    raise FileNotFoundError("Start Jupyter in the repository folder, or set INDOOR_AQ_CODE to that folder.")

CODE = locate_code_directory()

# Load the cells tagged "library" from 00_shared_functions.ipynb as an in-memory
# module named `shared`, so that `from shared import ...` works below.
# No helper .py file is generated.
_library_path = CODE / "00_shared_functions.ipynb"
_library = types.ModuleType("shared")
_library.__file__ = str(_library_path)
_library.__dict__["CODE"] = CODE
for _cell in json.loads(_library_path.read_text(encoding="utf-8"))["cells"]:
    if _cell["cell_type"] == "code" and "library" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), str(_library_path), "exec"), _library.__dict__)
sys.modules["shared"] = _library

from shared import DATA, output_root
print(f"Manuscript code: {CODE}")

## Configure this reproduction run

Choose the separate output directory. The default is `results/` in the repository folder; rebuilt files in that directory take precedence over supplied inputs. Use a new directory when testing a changed input.

In [ ]:
# Paths are shared by Python and R. Defaults: data/ for supplied inputs,
# results/ for this run. Set INDOOR_AQ_DATA and INDOOR_AQ_OUTPUT before
# starting Jupyter and R; relative paths are resolved against the repository.
OUTPUT = os.environ.get("INDOOR_AQ_OUTPUT") or None

## Import the required libraries

These imports belong to the active notebook kernel. Install `requirements.txt` in that same environment if an import fails.

In [ ]:
import numpy as np
import pandas as pd
from shared import load_hourly, load_daily, output_root, OUTCOMES, crosswalk, dump_json

## Load observations and define descriptive statistics

The hourly and daily files serve different purposes. Summaries omit missing values and retain their valid counts. No winsorization or imputation is added.

In [ ]:
out = output_root(OUTPUT)
tables_dir = out / 'tables'
x = load_hourly(out)   # hourly values with local date, clock hour and season
d = load_daily(out)    # daily means (at least 18 valid hours) with season


def summary(frame, value, keys):
    """Count, mean, median, quartiles and range of `value` for each group in `keys`."""
    return frame.groupby(keys)[value].agg(
        n='count', mean='mean', median='median',
        q25=lambda s: s.quantile(.25), q75=lambda s: s.quantile(.75),
        min='min', max='max').reset_index()

## Compute seasonal distributions and equal-home clock-hour profiles

For Figure 2, first compute each home's median for a given outcome, season, and clock hour, then take the median across homes. This gives each contributing home equal weight. Peak-to-trough amplitude is the difference between that home's maximum and minimum clock-hour medians.

In [ ]:
hourly = summary(x, 'value', ['home', 'variable'])
seasonal = summary(x, 'value', ['home', 'variable', 'season'])
seasonal_daily = summary(d, 'y', ['home', 'variable', 'season'])

# Step 1: each home's median for every outcome x season x clock hour.
home_hours = (x.dropna(subset=['value'])
              .groupby(['home', 'variable', 'season', 'hour'])
              .value.agg(home_median='median', n_valid='count')
              .reset_index())
# Step 2: median across homes, each home weighted equally (Figure 2).
profiles = (home_hours.groupby(['variable', 'season', 'hour'])
            .home_median.agg(median='median', n_homes='count')
            .reset_index())
# Within-home peak-to-trough amplitude of the clock-hour medians.
amplitudes = (home_hours.groupby(['home', 'variable', 'season'])
              .home_median.agg(min='min', max='max')
              .reset_index())
amplitudes['amplitude'] = amplitudes['max'] - amplitudes['min']

## Calculate monitoring availability and threshold burdens

Use daily PM2.5 means for the fraction above 15 micrograms/m3. Use hourly values for temperature at or below 18 degrees C, RH at or above 60%, and CO2 at or above 1,000 ppm. The denominator is the nonmissing data for that home and outcome; it is not the full calendar window.

In [ ]:
# Threshold burden = share of a home's valid data beyond the threshold:
#   PM2.5 : daily mean  > 15 ug/m3
#   temp  : hourly value <= 18 C
#   RH    : hourly value >= 60 %
#   CO2   : hourly value >= 1,000 ppm
availability = []
burdens = []
for home in crosswalk().home:
    for v in OUTCOMES:
        h = x[x.home.eq(home) & x.variable.eq(v) & x.value.notna()]   # valid hours
        a = d[d.home.eq(home) & d.variable.eq(v) & d.y.notna()]       # valid days
        availability.append(dict(
            home=home, variable=v, first_date=h.date.min(), last_date=h.date.max(),
            valid_hours=len(h), valid_days=len(a),
            heating_days=int(a.season.eq('Heating season').sum()),
            nonheating_days=int(a.season.eq('Non-heating season').sum())))

        values = a.y if v == 'pm25' else h.value
        if v == 'pm25':
            hits = values.gt(15)
        elif v == 'temp':
            hits = values.le(18)
        else:
            hits = values.ge(60 if v == 'rh' else 1000)
        burdens.append(dict(home=home, variable=v, n_valid=len(values), n_beyond=int(hits.sum()),
                            burden_pct=100 * hits.mean(), over_20_percent=bool(hits.mean() > .2)))

## Assemble Table S1 and non-heating PM2.5 groups

Monitoring dates and seasonal day counts in Table S1 follow the temperature availability series, as in the manuscript. Non-heating PM2.5 groups use each home's median of valid daily means, with boundaries at 15 and 50 micrograms/m3.

In [ ]:
availability = pd.DataFrame(availability)
burdens = pd.DataFrame(burdens)

# Table S1: monitoring dates and day counts from the temperature series plus
# the four threshold burdens; one row per home in numeric order.
burden_columns = burdens.pivot(index='home', columns='variable', values='burden_pct').add_suffix('_burden_pct')
s1 = (availability[availability.variable.eq('temp')]
      .drop(columns='variable')
      .merge(burden_columns, on='home', validate='one_to_one'))
s1['Home'] = s1.home.str.split().str[-1].astype(int)
s1 = s1.sort_values('Home')

# Non-heating PM2.5 group of each home, from the median of its valid daily means.
classifications = seasonal_daily[seasonal_daily.variable.eq('pm25')
                                 & seasonal_daily.season.eq('Non-heating season')].copy()
classifications['group'] = np.where(classifications['median'].le(15), '<=15',
                                    np.where(classifications['median'].le(50), '>15 to <=50', '>50'))

## Save descriptive outputs and verify reported counts

Write all summaries as unrounded CSV files. The numbers of homes exceeding 20% threshold burden must be 10 for temperature, 16 for RH, 20 for PM2.5, and 8 for CO2. These are descriptive counts, not estimated retrofit effects.

In [ ]:
frames = {
    'hourly_summary': hourly,
    'seasonal_hourly_summary': seasonal,
    'seasonal_daily_summary': seasonal_daily,
    'diurnal_home_medians': home_hours,
    'diurnal_equal_home_summary': profiles,
    'diurnal_amplitudes': amplitudes,
    'monitoring_availability': availability,
    'threshold_burdens': burdens,
    'Table_S1': s1,
    'PM25_nonheating_classification': classifications,
}
for name, frame in frames.items():
    frame.to_csv(tables_dir / (name + '.csv'), index=False)

# Number of homes with more than 20% of valid data beyond each threshold (Results 3.2).
counts = burdens.groupby('variable').over_20_percent.sum().astype(int).to_dict()
assert counts == dict(temp=10, rh=16, pm25=20, co2=8)

dump_json(out / 'logs/descriptive_checks.json', dict(
    homes=23,
    threshold_counts=counts,
    nonheating_PM25_groups=classifications.group.value_counts().to_dict(),
    daily_PM25_home3=int(availability.query("home == 'Home 3' and variable == 'pm25'").valid_days.iloc[0])))
print('Descriptive outputs rebuilt: 23 homes; threshold counts T/RH/PM2.5/CO2 = 10/16/20/8.')

## Interpret the outputs

Review the saved CSV files and JSON checks in the selected output directory. Keep the settings in `analysis_settings.json`
fixed when reproducing this manuscript. If a source-data assertion fails, inspect the named records before changing
any thresholds. `README.md` documents the order of all manuscript stages.